# 03 - Modeling

Trains and evaluates Logistic Regression and Random Forest on the processed Student Depression dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
import joblib

train = pd.read_csv('../data/processed/train.csv')
test = pd.read_csv('../data/processed/test.csv')

X_train = train.drop(columns=['Depression'])
y_train = train['Depression']
X_test = test.drop(columns=['Depression'])
y_test = test['Depression']

print('Train:', X_train.shape, '| Test:', X_test.shape)

Train: (22318, 39) | Test: (5580, 39)


## Scale features for Logistic Regression

Logistic Regression is sensitive to feature scale; Random Forest is not, so it will use the unscaled data. The scaler is fit on train only to avoid leakage.

In [2]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled[:3])

[[-1.12364183 -1.59883156 -0.10320629  1.56086291  1.50753851  0.53375086
   1.37559709  0.76005908 -1.39643049  0.59964819  1.03196506 -0.23595298
  -0.23627026 -0.26804307 -0.17402139 -0.20852416 -0.1473069  -0.16086985
  -0.23254743 -0.1473069  -0.1851869  -0.18140012  1.90912154 -0.15701756
  -0.13161133 -0.1634889  -0.17497567 -0.14967095 -0.19702687 -0.14214356
  -0.14490761 -0.15611641 -0.19825071 -0.14935772 -0.0822588  -0.08170488
  -0.21407809 -0.03544247 -0.13781763]
 [ 0.88996331 -1.395415    1.34910016 -1.14108564 -0.6942313  -1.24366659
   1.37559709  0.76005908  0.76426042  1.29566172  1.03196506 -0.23595298
  -0.23627026 -0.26804307 -0.17402139 -0.20852416 -0.1473069  -0.16086985
  -0.23254743 -0.1473069  -0.1851869  -0.18140012  1.90912154 -0.15701756
  -0.13161133 -0.1634889  -0.17497567 -0.14967095 -0.19702687 -0.14214356
  -0.14490761 -0.15611641 -0.19825071 -0.14935772 -0.0822588  -0.08170488
  -0.21407809 -0.03544247 -0.13781763]
 [ 0.88996331 -0.3783322   0.62294

# Train Model

In [3]:
log_reg = LogisticRegression(max_iter=100, random_state=42)
log_reg.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

print('Both models trained')

Both models trained


## Evaluate

In [7]:
def evaluate(name, y_true, y_pred, y_proba):
    print({name})
    print('Accuracy:  ', round(accuracy_score(y_true, y_pred), 4))
    print('Precision:',  round(precision_score(y_true, y_pred),4))
    print('Recall: ', round(recall_score(y_true, y_pred), 4))
    print('F1:       ', round(f1_score(y_true, y_pred), 4))
    print('ROC-AUC:  ', round(roc_auc_score(y_true, y_proba), 4))
    print()
    print('Confusion matrix:')
    print(confusion_matrix(y_true, y_pred))
    print()

log_reg_pred = log_reg.predict(X_test_scaled)
log_reg_proba = log_reg.predict_proba(X_test_scaled)[:, 1]
evaluate('Logistic Regression', y_test, log_reg_pred, log_reg_proba)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
evaluate('Random Forest', y_test, rf_pred, rf_proba)

{'Logistic Regression'}
Accuracy:   0.8464
Precision: 0.8563
Recall:  0.8864
F1:        0.8711
ROC-AUC:   0.9183

Confusion matrix:
[[1827  486]
 [ 371 2896]]

{'Random Forest'}
Accuracy:   0.8385
Precision: 0.8512
Recall:  0.8776
F1:        0.8642
ROC-AUC:   0.911

Confusion matrix:
[[1812  501]
 [ 400 2867]]



## Save models

In [8]:
joblib.dump(log_reg, '../models/logistic_regression.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(rf, '../models/random_forest.joblib')
print('Models saved to models/')

Models saved to models/


## Baseline: DummyClassifier

A naive baseline that ignores the features entirely, to confirm the real models are actually learning something beyond class imbalance.

In [9]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)
dummy_proba = dummy.predict_proba(X_test)[:, 1]
evaluate('Dummy (most frequent)', y_test, dummy_pred, dummy_proba)

{'Dummy (most frequent)'}
Accuracy:   0.5855
Precision: 0.5855
Recall:  1.0
F1:        0.7386
ROC-AUC:   0.5

Confusion matrix:
[[   0 2313]
 [   0 3267]]



## Cross-validation

5-fold stratified cross-validation on the training set, to check how stable each model's performance is beyond a single train/test split. Logistic Regression uses a Pipeline so scaling is refit correctly within each fold.

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

log_reg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=100, random_state=42)),
])

log_reg_cv = cross_validate(log_reg_pipeline, X_train, y_train, cv=cv, scoring=scoring)
rf_cv = cross_validate(rf, X_train, y_train, cv=cv, scoring=scoring)

for metric in scoring:
    lr_scores = log_reg_cv[f'test_{metric}']
    rf_scores = rf_cv[f'test_{metric}']
    print(f'{metric:10s} | LR: {lr_scores.mean():.4f} ± {lr_scores.std():.4f} | RF: {rf_scores.mean():.4f} ± {rf_scores.std():.4f}')

accuracy   | LR: 0.8466 ± 0.0051 | RF: 0.8425 ± 0.0041
precision  | LR: 0.8567 ± 0.0057 | RF: 0.8553 ± 0.0053
recall     | LR: 0.8862 ± 0.0061 | RF: 0.8801 ± 0.0049
f1         | LR: 0.8712 ± 0.0043 | RF: 0.8675 ± 0.0033
roc_auc    | LR: 0.9214 ± 0.0028 | RF: 0.9151 ± 0.0029


## Feature importance

Random Forest's built-in impurity-based importance, and Logistic Regression's coefficients (comparable across features since inputs were scaled).

In [13]:
rf_importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print('--- Random Forest: top 15 features ---')
print(rf_importance.head(15))

--- Random Forest: top 15 features ---
Have you ever had suicidal thoughts ?    0.209102
Academic Pressure                        0.164368
Financial Stress                         0.099633
CGPA                                     0.089509
Age                                      0.084654
Work/Study Hours                         0.075842
Study Satisfaction                       0.046178
Dietary Habits                           0.038659
Sleep Duration                           0.036329
Gender                                   0.017805
Family History of Mental Illness         0.015712
Degree_Class 12                          0.010631
Degree_BCA                               0.006094
Degree_B.Com                             0.006057
Degree_B.Ed                              0.006038
dtype: float64


In [14]:
lr_coef = pd.Series(log_reg.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False)
print('--- Logistic Regression: top 15 coefficients (by magnitude) ---')
print(lr_coef.head(15))

--- Logistic Regression: top 15 coefficients (by magnitude) ---
Have you ever had suicidal thoughts ?    1.217272
Academic Pressure                        1.154537
Financial Stress                         0.813222
Age                                     -0.595318
Work/Study Hours                         0.449454
Dietary Habits                          -0.440180
Study Satisfaction                      -0.331214
Sleep Duration                          -0.205747
Family History of Mental Illness         0.132658
CGPA                                     0.082582
Degree_Class 12                         -0.056635
Degree_LLB                               0.046817
Degree_B.Tech                            0.039688
Degree_MSc                              -0.037290
Degree_LLM                               0.031893
dtype: float64


## Final model selection: Logistic Regression

Logistic Regression outperformed Random Forest on every metric (Accuracy
84.64% vs 83.85%, Recall 88.64% vs 87.76%, F1 87.11% vs 86.42%, ROC-AUC
0.9183 vs 0.9110), and 5-fold cross-validation confirmed this margin is
stable, not due to a lucky split (both models had low standard deviation
across folds). Logistic Regression is also more interpretable: its
coefficients directly show both the magnitude and direction of each
feature's effect, which suits a project where explainability matters.
Selected as the final model.